# 01 — EML Operator Basics

This notebook introduces the EML (Exp-Minus-Log) operator and demonstrates its fundamental properties.

**Key result (Odrzywołek, 2026):** The operator `eml(x, y) = exp(x) - ln(y)`, paired with the constant 1, can generate **every** elementary function via composition.

Grammar: `S → 1 | eml(S, S)`

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '..')
from koopman_eml.eml_ops import eml, taylor_exp, taylor_ln

## 1. Taylor-Series Accuracy

Our `exp` and `ln` implementations use Horner-form Taylor series with range reduction.
Let's verify they match PyTorch's built-in functions.

In [ ]:
x = torch.linspace(-10, 10, 1000)
y = torch.linspace(0.01, 100, 1000)

exp_err = (taylor_exp(x) - torch.exp(x)).abs()
ln_err = (taylor_ln(y) - torch.log(y)).abs()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.semilogy(x.numpy(), exp_err.detach().numpy())
ax1.set_title('|taylor_exp(x) - exp(x)|')
ax1.set_xlabel('x')
ax1.set_ylabel('Absolute error')

ax2.semilogy(y.numpy(), ln_err.detach().numpy())
ax2.set_title('|taylor_ln(y) - ln(y)|')
ax2.set_xlabel('y')
ax2.set_ylabel('Absolute error')

plt.tight_layout()
plt.show()
print(f'Max exp error: {exp_err.max():.2e}')
print(f'Max ln error:  {ln_err.max():.2e}')

## 2. Fundamental EML Identities

Key identities from the EML paper:
- `eml(x, 1) = exp(x)` (exponential)
- `eml(1, 1) = e` (Euler's number)
- `eml(0, 1) = 1` (unit)
- `ln(x)` via depth-3 composition

In [ ]:
# eml(x, 1) = exp(x)
x = torch.linspace(-3, 3, 200)
ones = torch.ones_like(x)

result = eml(x, ones)
expected = torch.exp(x)

plt.figure(figsize=(8, 4))
plt.plot(x.numpy(), result.detach().numpy(), label='eml(x, 1)', linewidth=2)
plt.plot(x.numpy(), expected.numpy(), '--', label='exp(x)', linewidth=2)
plt.legend()
plt.title('Identity: eml(x, 1) = exp(x)')
plt.xlabel('x')
plt.show()

print(f'eml(1, 1) = {eml(torch.tensor([1.0]), torch.tensor([1.0])).item():.10f}')
print(f'e         = {np.e:.10f}')
print(f'eml(0, 1) = {eml(torch.tensor([0.0]), torch.tensor([1.0])).item():.10f}')

## 3. Deriving ln(x) from EML

Natural log requires a depth-3 EML tree:

```
ln(x) = eml(1, eml(eml(1, x), 1))
       = e - ln(exp(e - ln(x)))
       = e - (e - ln(x))
       = ln(x)  ✓
```

In [ ]:
x = torch.linspace(0.1, 10.0, 200)
ones = torch.ones_like(x)

# Build ln(x) from EML: eml(1, eml(eml(1, x), 1))
step1 = eml(ones, x)        # e - ln(x)
step2 = eml(step1, ones)    # exp(e - ln(x))
step3 = eml(ones, step2)    # e - ln(exp(e - ln(x))) = ln(x)

plt.figure(figsize=(8, 4))
plt.plot(x.numpy(), step3.detach().numpy(), label='eml(1, eml(eml(1,x),1))', linewidth=2)
plt.plot(x.numpy(), torch.log(x).numpy(), '--', label='ln(x)', linewidth=2)
plt.legend()
plt.title('Deriving ln(x) from EML (depth 3)')
plt.xlabel('x')
plt.show()

err = (step3 - torch.log(x)).abs().max()
print(f'Max error: {err.item():.2e}')

## 4. EML Tree Evaluation

Let's build a small EML tree and visualize its output.

In [ ]:
from koopman_eml.eml_tree import EMLTree

tree = EMLTree(depth=2, n_vars=1)
x = torch.linspace(-2, 2, 200).unsqueeze(-1)

with torch.no_grad():
    y_soft = tree(x, tau=1.0)

routes = tree.snap_weights()
print('Snapped routes:', routes)

with torch.no_grad():
    y_snapped = tree(x, tau=0.01)

plt.figure(figsize=(8, 4))
plt.plot(x[:, 0].numpy(), y_soft.numpy(), label='Soft (tau=1)', alpha=0.7)
plt.plot(x[:, 0].numpy(), y_snapped.numpy(), label='Snapped', linewidth=2)
plt.legend()
plt.title('EML Tree Output (depth=2, 1 variable)')
plt.xlabel('x')
plt.show()

## 5. Gradient Verification

The EML operator has well-defined gradients: ∂eml/∂x = exp(x) and ∂eml/∂y = -1/y.

In [ ]:
x = torch.tensor([1.0, 2.0, -1.0], requires_grad=True)
y = torch.tensor([1.0, 3.0, 0.5], requires_grad=True)

result = eml(x, y).sum()
result.backward()

print('x:     ', x.data.numpy())
print('dx:    ', x.grad.numpy())
print('exp(x):', torch.exp(x).detach().numpy())
print()
print('y:     ', y.data.numpy())
print('dy:    ', y.grad.numpy())
print('-1/y:  ', (-1.0/y).detach().numpy())